# Hands-On 8: Build Research, Analyst and Writer Crew

### Definition
A **multi-agent crew** divides a complex workflow among specialized agents. Each agent performs a focused task and passes its output to the next agent.

### Flow
`Research → Analysis → Writing → Review → Final Report`

In [1]:
from crewai import Agent, Task, Crew, Process
import os
from dotenv import load_dotenv

load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))
print("Model:", os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini"))
print("Base URL:", os.getenv("OPENAI_BASE_URL"))

llm = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")

OpenRouter key loaded: True
Model: openai/gpt-4o-mini
Base URL: https://openrouter.ai/api/v1


## 1. Create Specialized Agents

- **Researcher:** gathers information
- **Data Analyst:** identifies trends and insights
- **Writer:** produces the report
- **Reviewer:** checks quality and accuracy

In [2]:
researcher = Agent(
    role="Lead Research Specialist",
    goal="Conduct thorough research and provide reliable findings",
    backstory="An experienced researcher who gathers facts and organizes them clearly.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

data_analyst = Agent(
    role="Data Analyst",
    goal="Analyze research findings and identify important trends and insights",
    backstory="A strategic analyst skilled at turning research into actionable insights.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

writer = Agent(
    role="Professional Writer",
    goal="Create a clear, structured, and engaging final report",
    backstory="An experienced writer who converts complex information into readable reports.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

reviewer = Agent(
    role="Quality Reviewer",
    goal="Review generated content for accuracy, clarity, completeness, and consistency",
    backstory="A meticulous editor who identifies weaknesses and suggests improvements.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("All 4 agents created")

All 4 agents created


## 2. Define Tasks and Pass Outputs

`context=[previous_task]` passes the previous task output to the next agent.

In [3]:
research_task = Task(
    description=(
        "Research the current state of electric vehicles in 2025. Cover market growth, "
        "major manufacturers, battery technology, charging infrastructure, and key challenges."
    ),
    expected_output="A structured research summary with key facts and examples.",
    agent=researcher
)

analysis_task = Task(
    description=(
        "Analyze the research findings. Identify major trends, opportunities, challenges, "
        "and business implications for the EV industry."
    ),
    expected_output="A concise analysis of trends, opportunities, risks, and implications.",
    agent=data_analyst,
    context=[research_task]
)

writing_task = Task(
    description=(
        "Write a professional report on the EV industry using the research and analysis. "
        "Organize it with an introduction, key findings, trends, challenges, and conclusion."
    ),
    expected_output="A polished professional EV industry report.",
    agent=writer,
    context=[research_task, analysis_task]
)

review_task = Task(
    description=(
        "Review the generated report. Check whether it is complete, logically structured, "
        "clear, consistent with the research, and free from unsupported claims. "
        "Provide corrections and an improved final version."
    ),
    expected_output="A reviewed and improved final report.",
    agent=reviewer,
    context=[research_task, analysis_task, writing_task]
)

print("All tasks created")

All tasks created


## 3. Build the Collaboration Crew

Tasks execute sequentially and each task receives the required previous outputs.

In [4]:
collaboration_crew = Crew(
    agents=[researcher, data_analyst, writer, reviewer],
    tasks=[research_task, analysis_task, writing_task, review_task],
    process=Process.sequential,
    verbose=True,
    memory=True
)

print("Multi-agent collaboration crew built")
print("Agents:", len(collaboration_crew.agents))
print("Tasks:", len(collaboration_crew.tasks))

Multi-agent collaboration crew built
Agents: 4
Tasks: 4


## 4. Execute the Workflow

In [5]:
print("=" * 70)
print("EXECUTING MULTI-AGENT COLLABORATION")
print("=" * 70)

result = await collaboration_crew.kickoff_async()

print("\n" + "=" * 70)
print("FINAL REPORT")
print("=" * 70)
print(result)

EXECUTING MULTI-AGENT COLLABORATION


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f59b22e5-fbd8-4402-bd27-357d0c3b5536                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the current state of electric vehicles in 2025. Cover market growth, major manufacturers,       │
│  battery technology, charging infrastructure, and key challenges.                                               │
│  ID: d4619483-c9d7-4485-a3c9-00a82863839d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Research Specialist                                                                                │
│                                                                                                                 │
│  Task: Research the current state of electric vehicles in 2025. Cover market growth, major manufacturers,       │
│  battery technology, charging infrastructure, and key challenges.                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['electric vehicles market growth 2025', 'major manufacturers of electric vehicles 2025',    │
│  'battery technology electric vehicles 2025', 'charging infrastructure electric vehicles 2025', 'ch...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['current state of electric vehicles 2025']}                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (35 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the generated report. Check whether it is complete, logically structured, clear, consistent with  │
│  the research, and free from unsupported claims. Provide corrections and an improved final version.             │
│  Agent: Quality Reviewer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f59b22e5-fbd8-4402-bd27-357d0c3b5536                                                                       │
│  Final Output: # Electric Vehicle Industry Report 2025                                                          │
│                                                                                                                 │
│  ## Introduction                                                                                                │
│  As of 2025, the electric vehicle (EV) industry is undergoing a remarkable transformation. The shift towards    │
│  sustainable transportation is evident through extensive market growth, technological innovations in battery    │
│  production, and advancements in charging infrastructure. This report delves into the current state of the EV   │
│  industry, highlighting key findings, prevailing trends, challenges faced, and the implications for             │
│  stakeholders in the automotive ecosystem.                                                                      │
│                                                                                                                 │
│  ## Key Findings                                                                                                │
│                                                                                                                 │
│  ### 1. Market Growth                                                                                           │
│  The global electric vehicle market has experienced unprecedented growth, with expected annual sales exceeding  │
│  30 million units and capturing over 25% of total vehicle sales. This growth is propelled by heightened         │
│  consumer demand for environmentally friendly modes of transportation, government incentives, and strategic     │
│  investments by leading manufacturers.                                                                          │
│                                                                                                                 │
│  ### 2. Dominant Manufacturers                                                                                  │
│  The EV landscape is marked by several industry stalwarts:                                                      │
│  - **Tesla** remains at the forefront, recognized for its expansive range of models and an extensive charging   │
│  network.                                                                                                       │
│  - **BYD** is now the largest electric vehicle manufacturer globally, showcasing strong production              │
│  capabilities.                                                                                                  │
│  - **Volkswagen** has committed substantial resources to EV technology, seeking a top position in the global    │
│  market.                                                                                                        │
│  - **General Motors (GM)** and **Ford** have made significant strides too, expanding their electric portfolios  │
│  with popular models like the Chevrolet Bolt and the all-electric F-150 Lightning.                              │
│                                                                                                                 │
│  ### 3. Advancements in Battery Technology                                                                      │
│  Technological improvements in battery production are 


FINAL REPORT
# Electric Vehicle Industry Report 2025

## Introduction
As of 2025, the electric vehicle (EV) industry is undergoing a remarkable transformation. The shift towards sustainable transportation is evident through extensive market growth, technological innovations in battery production, and advancements in charging infrastructure. This report delves into the current state of the EV industry, highlighting key findings, prevailing trends, challenges faced, and the implications for stakeholders in the automotive ecosystem.

## Key Findings

### 1. Market Growth
The global electric vehicle market has experienced unprecedented growth, with expected annual sales exceeding 30 million units and capturing over 25% of total vehicle sales. This growth is propelled by heightened consumer demand for environmentally friendly modes of transportation, government incentives, and strategic investments by leading manufacturers.

### 2. Dominant Manufacturers
The EV landscape is marked by sever

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 5. Inspect Individual Outputs

In [6]:
tasks = [
    research_task,
    analysis_task,
    writing_task,
    review_task
]

for task in tasks:
    print("\n" + "=" * 60)
    print(task.agent.role)
    print("=" * 60)

    if task.output:
        print(task.output.raw)
    else:
        print("No output available")


Lead Research Specialist
### Current State of Electric Vehicles in 2025

**1. Market Growth**
The electric vehicle (EV) market has experienced significant growth leading into 2025. By this year, it is estimated that global EV sales will have surged past 30 million units annually, representing a market share of over 25% of total vehicle sales. This increase can be attributed to various factors including government incentives, growing consumer demand for environmentally friendly transport, and the increasing focus on sustainability within the automotive industry.

**2. Major Manufacturers**
In 2025, several key manufacturers are dominating the EV sector:
- **Tesla** continues to lead with its broad range of models and extensive charging network.
- **Volkswagen** has made substantial investments in EV production and technology, aiming to become a global leader in electric cars.
- **General Motors** (GM) has expanded its lineup of electric models, including the popular Chevrolet Bolt and 

## 6. Collaboration Flow

```text
Researcher
    ↓
Research Findings
    ↓
Data Analyst
    ↓
Trends + Insights
    ↓
Writer
    ↓
Draft Report
    ↓
Reviewer
    ↓
Final Report
```

### Key principle
Specialized agents work independently on their responsibilities, while `context` connects their outputs.

## 7. Final Verification

- ✅ Research agent created
- ✅ Data analyst created
- ✅ Writer agent created
- ✅ Review agent created
- ✅ Tasks assigned
- ✅ Outputs passed between agents
- ✅ Generated content reviewed
- ✅ Final report produced
- ✅ Crew memory enabled